In [ ]:
# =============================================================================
# TASK 1: Graph Concepts & State Design
# =============================================================================
# LangGraph builds agents as GRAPHS instead of flat loops.
#
# CORE CONCEPTS:
# - StateGraph: Container for your workflow (like a flowchart)
# - State: Shared TypedDict that flows through all nodes
# - Node: A function that reads state, does work, returns updated fields
# - Edge: Transition to the next node (add_edge)
# - Conditional Edge: "If X go here, if Y go there" (add_conditional_edges)
# - Entry Point: Where execution starts (set_entry_point)
# - END: Special node that stops the graph
# =============================================================================

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
import operator

# --- STATE SCHEMA ---
# Defines what data flows through the entire graph.
# Every node reads from this and writes to this.
class ResearchState(TypedDict):
    query: str                          # What the user asked
    plan: str                           # What the agent plans to do
    search_results: str                 # Raw results from tools
    answer: str                         # Generated answer
    quality_score: int                  # Score 0-10 (10 = perfect)
    retry_count: int                    # How many times we retried
    max_retries: int                    # Safety limit

# --- NODES ---
# Each node: takes state, returns dict of updated fields

def plan_node(state):
    """Create a plan based on the query."""
    query = state["query"]
    plan = f"Search for '{query}', then summarize findings"
    return {"plan": plan}

def execute_node(state):
    """Execute the plan (simulate search)."""
    results = f"Found info about: {state['query']}. LangGraph is a graph-based agent framework."
    return {"search_results": results}

def generate_node(state):
    """Generate an answer from search results."""
    answer = f"Based on research: {state['search_results']}. LangGraph gives you branching and self-correction."
    return {"answer": answer}

def critique_node(state):
    """Critique the answer and assign quality score."""
    retry = state["retry_count"]
    # First attempt: low score. Retry: high score.
    score = 5 if retry == 0 else 8
    return {"quality_score": score}

# --- CONDITIONAL ROUTER ---
# Decides: finish or retry?

def route_after_critique(state):
    score = state["quality_score"]
    retries = state["retry_count"]
    max_retries = state["max_retries"]
    if score >= 7:
        return "finish"
    elif retries < max_retries:
        return "retry"
    else:
        return "finish"

# --- BUILD THE GRAPH ---
graph = StateGraph(ResearchState)

# Add nodes
graph.add_node("plan", plan_node)
graph.add_node("execute", execute_node)
graph.add_node("generate", generate_node)
graph.add_node("critique", critique_node)

# Add linear edges
graph.add_edge("plan", "execute")
graph.add_edge("execute", "generate")
graph.add_edge("generate", "critique")

# Add conditional edge (this is the "retry or finish" logic)
graph.add_conditional_edges("critique", route_after_critique, {
    "retry": "execute",     # if retry -> go back to execute
    "finish": END           # if finish -> stop
})

# Set where execution starts
graph.set_entry_point("plan")

# Compile (makes it runnable)
app = graph.compile()

# --- RUN IT ---
print("GRAPH BUILT SUCCESSFULLY")
print("=" * 60)
result = app.invoke({
    "query": "What is LangGraph?",
    "plan": "",
    "search_results": "",
    "answer": "",
    "quality_score": 0,
    "retry_count": 0,
    "max_retries": 3,
})

print(f"\nFINAL ANSWER: {result['answer'][:80]}...")
print(f"QUALITY SCORE: {result['quality_score']}/10")
print(f"RETRIES: {result['retry_count']}")

SIMULATING THE GRAPH (before building it)

--- Pass 1 ---
  [PLAN] Created plan: Search for 'What is LangGraph?', then summarize findings
  [EXECUTE] Got results: Found info about: What is LangGraph?. Key facts: L...
  [GENERATE] Generated answer: Based on research: Found info about: What is LangG...
  [CRITIQUE] Score: 5/10 (attempt #1)
  [ROUTE] Score 5 < 7 and retries 0 < 3 -> RETRY

--- Pass 2 (retry #1) ---
  [EXECUTE] Got results: Found info about: What is LangGraph?. Key facts: L...
  [GENERATE] Generated answer: Based on research: Found info about: What is LangG...
  [CRITIQUE] Score: 8/10 (attempt #2)
  [ROUTE] Score 8 >= 7 -> FINISH

FINAL STATE
  Query: What is LangGraph?
  Answer: Based on research: Found info about: What is LangGraph?. Key...
  Quality: 8/10
  Retries: 1
  Decision: finish
